## From P14 to "Simply Lovely" — 2022 Belgian GP Analysis

Last Updated: 2026-03-17

## Table of Contents
- [Section 1 — Setup & Data Load](#section-1)
- [Section 2 — Race Overview](#section-2)
- [Section 3 — Descriptive Statistics](#section-3)
- [Section 4 — Normal Distribution & Z-Score](#section-4)
- [Section 5 — Tire Degradation Regression](#section-5)
- [Section 6 — Pearson Correlation](#section-6)
- [Section 7 — Telemetry Analysis](#section-7)
- [Section 8 — Weather Context](#section-8)
- [Section 9 — Conclusion](#section-9)

<a id="section-1"></a>
## Section 1 — Setup & Data Load

This notebook builds a data-driven analysis of the **2022 Belgian Grand Prix** at Spa-Francorchamps.

Max Verstappen (VER) started from **P14** on the grid and still won the race. The central argument of this project is:

> His victory was **statistically inevitable**, given his pace and consistency relative to the field.

In this first section we will:
- Set up the Python environment and enable the FastF1 cache
- Load the 2022 Belgian GP race session
- Export cleaned laps, weather, results, and key telemetry for four drivers: **VER, PER, SAI, RUS**

In [4]:
# Data handling and numerical computing
import pandas as pd              # Work with tabular data (DataFrames)
import numpy as np               # Numerical operations and arrays

# Plotting and visualization (used later in the notebook)
import matplotlib.pyplot as plt  # Base plotting library
import seaborn as sns            # High-level statistical plotting built on matplotlib

# Statistical functions
from scipy import stats          # Hypothesis tests, distributions, etc.

# FastF1: main library for F1 timing and telemetry data
import fastf1

# Standard library
import os                        # Work with folders and file paths

# ---------------------------------------------------------------
# Enable on-disk cache for FastF1 so that data is downloaded only
# once and then reused from the local './cache' folder.
# ---------------------------------------------------------------
fastf1.Cache.enable_cache('./cache')

# ---------------------------------------------------------------
# Load the 2022 Belgian Grand Prix RACE session.
# - 2022      -> season year
# - 'Belgium' -> event name accepted by FastF1
# - 'R'       -> session type ('R' = Race)
# ---------------------------------------------------------------
session = fastf1.get_session(2022, 'Belgium', 'R')

# ---------------------------------------------------------------
# Load all timing and telemetry data for this race session.
# telemetry=True ensures that per-lap telemetry is available later.
# The first run may take a while because data is downloaded.
# ---------------------------------------------------------------
session.load(telemetry=True)

# ---------------------------------------------------------------
# Extract a few basic pieces of session information:
# - Event name
# - Event date
# - Total number of race laps
#
# session.event behaves like a Series with metadata, and
# session.laps is a DataFrame with one row per lap.
# ---------------------------------------------------------------
event_name = session.event['EventName']             # Human-readable event name
event_date = session.event['EventDate']             # Date of the race
total_laps = session.laps['LapNumber'].max()        # Highest lap number = total laps

print(f"Event      : {event_name}")
print(f"Date       : {event_date}")
print(f"Total laps : {int(total_laps)}")

core           INFO 	Loading data for Belgian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info


req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '10'
core        WARNING 	Fixed incorrect tyre stint information for driver '22'
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '63', '14', '16', '31', '5', '10', '23', '18', '4', '22', '24', '3', '20', '47', '6', '77', '44']


Event      : Belgian Grand Prix
Date       : 2022-08-28 00:00:00
Total laps : 44


In [5]:
# ---------------------------------------------------------------
# Make sure the './data' folder exists for all exported CSV files.
# exist_ok=True means "create if missing, do nothing if it exists".
# ---------------------------------------------------------------
data_dir = './data'
os.makedirs(data_dir, exist_ok=True)

# ---------------------------------------------------------------
# Helper function: convert a timedelta column (e.g. LapTime or Time)
# into a pure float column measured in seconds.
#
# Parameters:
# - df          : input DataFrame
# - time_col    : name of the timedelta column to convert
# - new_col     : name of the new float (seconds) column
# - drop_original: if True, remove the original timedelta column
#
# This guarantees lap/time measures are always stored as floats (seconds).
# ---------------------------------------------------------------
def convert_timedelta_to_seconds(df: pd.DataFrame,
                                 time_col: str,
                                 new_col: str,
                                 drop_original: bool = True) -> pd.DataFrame:
    df = df.copy()                                    # Avoid changing the original DataFrame in-place
    df[new_col] = df[time_col].dt.total_seconds()    # Convert Timedelta values to float seconds
    if drop_original:
        df = df.drop(columns=[time_col])             # Remove the original timedelta column if requested
    return df

In [6]:
# ---------------------------------------------------------------
# Load all laps for the session. After session.load(telemetry=True),
# session.laps is already a DataFrame with one row per lap.
# ---------------------------------------------------------------
laps_all = session.laps

# ---------------------------------------------------------------
# Keep ONLY the four drivers of interest for this project:
# VER (Verstappen), PER (Pérez), SAI (Sainz), RUS (Russell).
# The 'Driver' column holds the three-letter driver codes.
# ---------------------------------------------------------------
drivers_of_interest = ['VER', 'PER', 'SAI', 'RUS']
laps_filtered = laps_all[laps_all['Driver'].isin(drivers_of_interest)]

# ---------------------------------------------------------------
# FastF1 stores lap times in the 'LapTime' column as Timedelta.
# To follow the rule "lap time must always be seconds (float)",
# we convert 'LapTime' into 'LapTimeSeconds' and drop 'LapTime'.
# ---------------------------------------------------------------
if 'LapTime' in laps_filtered.columns:
    laps_filtered = convert_timedelta_to_seconds(
        df=laps_filtered,
        time_col='LapTime',
        new_col='LapTimeSeconds',
        drop_original=True
    )

# ---------------------------------------------------------------
# Save the filtered laps to './data/laps_data.csv'.
# index=False avoids adding the DataFrame index as an extra column.
# ---------------------------------------------------------------
laps_csv_path = os.path.join(data_dir, 'laps_data.csv')
laps_filtered.to_csv(laps_csv_path, index=False)
print(f"Saved laps data to {laps_csv_path}")

Saved laps data to ./data\laps_data.csv


In [7]:
# ---------------------------------------------------------------
# WEATHER DATA
#
# session.weather_data is a DataFrame with time-series weather
# samples (track temp, air temp, rain, etc.).
# We export it directly to CSV.
# ---------------------------------------------------------------
weather_df = session.weather_data

weather_csv_path = os.path.join(data_dir, 'weather_data.csv')
weather_df.to_csv(weather_csv_path, index=False)
print(f"Saved weather data to {weather_csv_path}")

# ---------------------------------------------------------------
# SESSION RESULTS
#
# session.results is a DataFrame with the race classification:
# finishing positions, points, teams, etc.
# We also export it directly to CSV.
# ---------------------------------------------------------------
results_df = session.results

results_csv_path = os.path.join(data_dir, 'session_results.csv')
results_df.to_csv(results_csv_path, index=False)
print(f"Saved session results to {results_csv_path}")

Saved weather data to ./data\weather_data.csv
Saved session results to ./data\session_results.csv


In [8]:
# ---------------------------------------------------------------
# For each of the four drivers (VER, PER, SAI, RUS), we:
# 1. Select all their laps from the session.
# 2. Find their single fastest lap.
# 3. Get telemetry for that lap (speed, throttle, etc.).
# 4. Convert the telemetry 'Time' column from Timedelta to seconds.
# 5. Save that telemetry to a driver-specific CSV file.
# ---------------------------------------------------------------

def save_fastest_lap_telemetry(session, driver_code: str, data_dir: str) -> None:
    # All laps for this driver only
    driver_laps = session.laps.pick_driver(driver_code)

    # Fastest lap object for this driver
    fastest_lap = driver_laps.pick_fastest()

    # Telemetry for this fastest lap; add_distance() computes distance
    # along the lap (useful x-axis for future charts).
    car_data = fastest_lap.get_car_data().add_distance()

    # Convert telemetry time column (Timedelta) into seconds (float).
    # In FastF1, telemetry time lives in the 'Time' column.
    if 'Time' in car_data.columns:
        car_data = convert_timedelta_to_seconds(
            df=car_data,
            time_col='Time',
            new_col='TimeSeconds',
            drop_original=True
        )

    # Build a filename like 'telemetry_VER.csv'
    filename = f"telemetry_{driver_code}.csv"
    filepath = os.path.join(data_dir, filename)

    # Export telemetry to CSV
    car_data.to_csv(filepath, index=False)

    # Print confirmation for this driver
    print(f"Saved fastest-lap telemetry for {driver_code} to {filepath}")

# ---------------------------------------------------------------
# Generate and save telemetry CSVs for exactly these four drivers.
# ---------------------------------------------------------------
for driver_code in ['VER', 'PER', 'SAI', 'RUS']:
    save_fastest_lap_telemetry(session, driver_code, data_dir)

Saved fastest-lap telemetry for VER to ./data\telemetry_VER.csv
Saved fastest-lap telemetry for PER to ./data\telemetry_PER.csv
Saved fastest-lap telemetry for SAI to ./data\telemetry_SAI.csv
Saved fastest-lap telemetry for RUS to ./data\telemetry_RUS.csv


e:\F12022\.venv\Lib\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
e:\F12022\.venv\Lib\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
e:\F12022\.venv\Lib\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
e:\F12022\.venv\Lib\site-packages\fastf1\core.py:3129: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"


In [9]:
# Simple final message so it's clear that all exports completed.
print("All data loaded and saved. Ready for analysis.")

All data loaded and saved. Ready for analysis.


## Caching note — faster future runs

All raw timing and telemetry data for the **2022 Belgian Grand Prix** is now cached locally in the `./cache` folder.

The first run may take longer because FastF1 has to download the data, but future runs of this notebook will load the same session **directly from the local cache**, making this setup step **much faster and more reliable**.

In [10]:
# ---------------------------------------------------------------
# SECTION 2 — Race Overview
# CHART 1: Position vs Lap Number for VER, PER, SAI, RUS
# ---------------------------------------------------------------

import pandas as pd                  # Ensure pandas is available in this cell
import matplotlib.pyplot as plt      # Plotting library
import seaborn as sns                # For consistent styling (optional but handy)
import os                            # For file paths

# Use a dark background style for all plots in this section
plt.style.use('dark_background')

# ---------------------------------------------------------------
# Load the laps data that we saved in Section 1.
# This CSV should be in the './data' folder and already restricted
# to the four drivers of interest (VER, PER, SAI, RUS).
# ---------------------------------------------------------------
laps_path = './data/laps_data.csv'
laps_df = pd.read_csv(laps_path)

# ---------------------------------------------------------------
# Make sure the LapNumber and Position columns are numeric so that
# matplotlib can plot them correctly.
# errors='coerce' will set invalid values to NaN (and we drop them).
# ---------------------------------------------------------------
laps_df['LapNumber'] = pd.to_numeric(laps_df['LapNumber'], errors='coerce')
laps_df['Position'] = pd.to_numeric(laps_df['Position'], errors='coerce')

# Drop any rows where LapNumber or Position is missing
laps_df = laps_df.dropna(subset=['LapNumber', 'Position'])

# ---------------------------------------------------------------
# Define fixed driver colors as requested.
# Keys are driver codes; values are hex color codes.
# ---------------------------------------------------------------
driver_colors = {
    'VER': '#E10600',  # Red Bull (Max Verstappen)
    'PER': '#0066CC',  # Red Bull (Sergio Pérez)
    'SAI': '#FF8000',  # Ferrari (Carlos Sainz)
    'RUS': '#C0C0C0',  # Mercedes (George Russell)
}

# ---------------------------------------------------------------
# Create the figure and axis for the race progression chart.
# figsize is in inches: width=14, height=6.
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 6))

# ---------------------------------------------------------------
# Plot a line for each driver: Position vs LapNumber.
# We loop over the four driver codes and filter laps for each one.
# ---------------------------------------------------------------
for driver_code, color in driver_colors.items():
    driver_laps = laps_df[laps_df['Driver'] == driver_code]  # Only laps for this driver
    
    # Plot LapNumber on x-axis and Position on y-axis
    ax.plot(
        driver_laps['LapNumber'],
        driver_laps['Position'],
        label=driver_code,
        color=color,
        linewidth=2.0
    )

# ---------------------------------------------------------------
# Invert the y-axis so that P1 appears at the top and higher
# position numbers (e.g. P20) are lower on the chart.
# ---------------------------------------------------------------
ax.invert_yaxis()

# ---------------------------------------------------------------
# Axis labels and title.
# ---------------------------------------------------------------
ax.set_xlabel('Lap Number')
ax.set_ylabel('Race Position (P)')
ax.set_title('Race Progression — 2022 Belgian Grand Prix')

# ---------------------------------------------------------------
# Add a light dashed grid to help read the chart.
# alpha controls transparency; linestyle='--' makes dashed lines.
# ---------------------------------------------------------------
ax.grid(alpha=0.2, linestyle='--')

# ---------------------------------------------------------------
# Add a legend to identify which line belongs to which driver.
# loc='upper right' places it in the top-right corner.
# ---------------------------------------------------------------
ax.legend(title='Driver', loc='upper right')

# ---------------------------------------------------------------
# Annotation 1: Highlight Verstappen starting P14 on lap 1.
#
# We:
# - Filter laps for VER
# - Pick the row where LapNumber == 1
# - Use its Position value for the annotation
# ---------------------------------------------------------------
ver_laps = laps_df[laps_df['Driver'] == 'VER']
ver_lap1 = ver_laps[ver_laps['LapNumber'] == 1]

if not ver_lap1.empty:
    ver_start_pos = ver_lap1['Position'].iloc[0]  # P14 at race start (should be 14)
    
    # Add an annotation with an arrow pointing to that first data point
    ax.annotate(
        text=f'VER starts P{int(ver_start_pos)}',
        xy=(1, ver_start_pos),                # Point to lap 1, starting position
        xytext=(3, ver_start_pos + 4),       # Place text a bit to the right and lower
        arrowprops=dict(arrowstyle='->', color='white'),
        color='white',
        fontsize=10
    )

# ---------------------------------------------------------------
# Annotation 2: First lap where VER reaches P1.
#
# We:
# - Look at VER's laps
# - Find the first lap where Position == 1
# - Annotate that point if it exists
# ---------------------------------------------------------------
ver_p1_laps = ver_laps[ver_laps['Position'] == 1]

if not ver_p1_laps.empty:
    first_p1_lap = ver_p1_laps['LapNumber'].min()           # Earliest lap in P1
    first_p1_pos = 1                                        # Position is P1
    
    ax.annotate(
        text=f'VER reaches P1 (Lap {int(first_p1_lap)})',
        xy=(first_p1_lap, first_p1_pos),                    # Point to P1 on that lap
        xytext=(first_p1_lap + 3, first_p1_pos + 5),        # Place text slightly to the right and lower
        arrowprops=dict(arrowstyle='->', color='white'),
        color='white',
        fontsize=10
    )

# ---------------------------------------------------------------
# Add a simple watermark in the bottom-right corner of the axes.
# transform=ax.transAxes lets us use relative coordinates:
# (1.0, 0.0) means right-bottom of the plotting area.
# ---------------------------------------------------------------
ax.text(
    1.0,
    0.0,
    'Data: FastF1 | 2022 Belgian GP',
    transform=ax.transAxes,
    ha='right',
    va='bottom',
    fontsize=9,
    color='white',
    alpha=0.7
)

# ---------------------------------------------------------------
# Save the figure to the './charts' folder at 150 DPI resolution.
# bbox_inches='tight' trims extra whitespace around the figure.
# ---------------------------------------------------------------
charts_dir = './charts'
os.makedirs(charts_dir, exist_ok=True)  # Ensure charts folder exists
race_progression_path = os.path.join(charts_dir, 'race_progression.png')
plt.savefig(race_progression_path, dpi=150, bbox_inches='tight')

print(f"Saved race progression chart to {race_progression_path}")

# Close the figure to free memory and avoid overlapping with future plots
plt.close(fig)

Saved race progression chart to ./charts\race_progression.png


In [11]:
# ---------------------------------------------------------------
# CHART 2: Final Results Table
#
# We will:
# - Load 'session_results.csv' from the './data' folder.
# - Build a matplotlib table (not just print the DataFrame).
# - Color rows based on team:
#     Red Bull = #1E3A5F
#     Ferrari  = #8B0000
#     Mercedes = #1A3A2A
# - Save the figure as './charts/final_results_table.png'.
# ---------------------------------------------------------------

import pandas as pd
import matplotlib.pyplot as plt
import os

plt.style.use('dark_background')  # Keep dark theme consistent

# ---------------------------------------------------------------
# Load session results exported in Section 1.
# ---------------------------------------------------------------
results_path = './data/session_results.csv'
results_df = pd.read_csv(results_path)

# ---------------------------------------------------------------
# Select and rename the columns we care about for the table.
#
# FastF1 column names can vary by version. In your export, the key
# columns are typically:
# - 'Position'
# - 'Abbreviation' (driver code like VER)
# - 'TeamName'
# - 'GridPosition'
# - 'Time' (winner has full race time; others are gaps)
# ---------------------------------------------------------------

# Helper: pick the first column name that exists in the DataFrame
def pick_col(df: pd.DataFrame, candidates: list[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of these columns were found: {candidates}. Available: {list(df.columns)}")

pos_col = pick_col(results_df, ['Position'])
driver_col = pick_col(results_df, ['Abbreviation', 'Driver', 'DriverCode'])
team_col = pick_col(results_df, ['TeamName', 'Team', 'Constructor'])
grid_col = pick_col(results_df, ['GridPosition', 'Grid'])
time_col = pick_col(results_df, ['Time', 'GapToLeader'])

table_df = results_df[[pos_col, driver_col, team_col, grid_col, time_col]].copy()

table_df = table_df.rename(columns={
    pos_col: 'Position',
    driver_col: 'Driver',
    team_col: 'Team',
    grid_col: 'Start Grid',
    time_col: 'Gap to Leader',
})

# ---------------------------------------------------------------
# Define background colors for teams of interest.
# All other teams default to a neutral dark grey.
# ---------------------------------------------------------------
team_colors = {
    'Red Bull Racing': '#1E3A5F',
    'Red Bull': '#1E3A5F',         # Extra key in case the name is shorter
    'Ferrari': '#8B0000',
    'Mercedes': '#1A3A2A',
}

default_row_color = '#222222'      # Neutral dark grey for other teams

# ---------------------------------------------------------------
# Build a list of row colors, one per row in the table, based on the team.
# ---------------------------------------------------------------
row_colors = []
for _, row in table_df.iterrows():
    team_name = row['Team']
    row_color = team_colors.get(team_name, default_row_color)  # Use default if team not listed
    row_colors.append(row_color)

# ---------------------------------------------------------------
# Create a figure for the table. We hide the axes and only show the table.
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 8))
ax.axis('off')  # Turn off the x and y axis

# ---------------------------------------------------------------
# Create the table:
# - cellText: values from the DataFrame
# - colLabels: column names for the header row
# - loc='center': center the table in the figure
# ---------------------------------------------------------------
table = ax.table(
    cellText=table_df.values,
    colLabels=table_df.columns,
    loc='center',
    cellLoc='center'
)

# ---------------------------------------------------------------
# Apply background colors to each data row (skip the header row).
# The header is row index 0 inside the table object, so data rows
# start from 1.
# ---------------------------------------------------------------
for i, color in enumerate(row_colors, start=1):
    for j in range(len(table_df.columns)):
        table[(i, j)].set_facecolor(color)

# ---------------------------------------------------------------
# Make header cells stand out with a slightly lighter color and bold text.
# ---------------------------------------------------------------
for j in range(len(table_df.columns)):
    header_cell = table[(0, j)]
    header_cell.set_facecolor('#333333')
    header_cell.set_text_props(weight='bold')

# ---------------------------------------------------------------
# Adjust font size and scaling so the table fits nicely in the figure.
# ---------------------------------------------------------------
table.auto_set_font_size(False)  # Disable automatic font resizing
table.set_fontsize(10)           # Set a fixed font size
table.scale(1.0, 1.4)            # Stretch table vertically a bit

# ---------------------------------------------------------------
# Add a title above the table.
# ---------------------------------------------------------------
ax.set_title(
    '2022 Belgian GP — Final Classification',
    fontsize=14,
    pad=20
)

# ---------------------------------------------------------------
# Save the table figure to the './charts' folder.
# ---------------------------------------------------------------
charts_dir = './charts'
os.makedirs(charts_dir, exist_ok=True)
final_table_path = os.path.join(charts_dir, 'final_results_table.png')
plt.savefig(final_table_path, dpi=150, bbox_inches='tight')

print(f"Saved final results table to {final_table_path}")

plt.close(fig)

KeyError: "['Driver', 'Team'] not in index"

<a id="section-2"></a>
## Section 2 — Race Overview

The race progression chart shows how quickly Verstappen climbed from **P14** on lap 1 through the field, overtaking cars almost every lap in the early phase. The annotation marks the first lap where he reaches **P1**, highlighting the moment his charge through the pack is complete and the race effectively comes under his control. This sets up the later statistical sections, where we will quantify how his pace, consistency, tire management, and telemetry traces made this climb not just impressive, but **statistically inevitable**.

In [ ]:
# ---------------------------------------------------------------
# SECTION 3 — Descriptive Statistics (Clean Lap Times)
# ---------------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Use a dark background style for charts in this section
plt.style.use('dark_background')

# ---------------------------------------------------------------
# 1. LOAD LAPS FROM CSV
# ---------------------------------------------------------------

laps_path = './data/laps_data.csv'          # Path to laps exported in Section 1
laps = pd.read_csv(laps_path)               # Read laps into a DataFrame

# Make sure numeric columns are actually numeric (in case CSV parsing changed them)
laps['LapNumber'] = pd.to_numeric(laps['LapNumber'], errors='coerce')

# ---------------------------------------------------------------
# 2. ENSURE LAP TIME IS IN SECONDS (FLOAT)
# ---------------------------------------------------------------
# Section 1 already created a 'LapTimeSeconds' column, but we guard
# against the case where it is missing by converting from 'LapTime'.
# ---------------------------------------------------------------

if 'LapTimeSeconds' not in laps.columns and 'LapTime' in laps.columns:
    # If we only have 'LapTime' (string), try to parse and convert
    laps['LapTime'] = pd.to_timedelta(laps['LapTime'], errors='coerce')
    laps['LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()

# Drop any rows where lap time in seconds is missing
laps = laps.dropna(subset=['LapTimeSeconds'])

# ---------------------------------------------------------------
# 3. FILTER OUT NON-CLEAN LAPS
# ---------------------------------------------------------------
# We remove:
# - Pit in laps      -> PitInTime is not null
# - Pit out laps     -> PitOutTime is not null
# - First lap        -> LapNumber == 1
# - Laps more than 2 SD ABOVE each driver's own mean
# ---------------------------------------------------------------

# Treat any literal 'NaN'/empty strings as missing
for col in ['PitInTime', 'PitOutTime']:
    if col in laps.columns:
        laps[col] = laps[col].replace(['', 'NaT', 'nan'], np.nan)

# Remove pit in and pit out laps (keep rows where both are null or column missing)
if 'PitInTime' in laps.columns:
    laps = laps[laps['PitInTime'].isna()]
if 'PitOutTime' in laps.columns:
    laps = laps[laps['PitOutTime'].isna()]

# Remove first lap of the race for all drivers
laps = laps[laps['LapNumber'] != 1]

# ---------------------------------------------------------------
# Remove laps that are more than 2 standard deviations ABOVE
# each driver's own mean lap time. We do this group-by-group:
# for each driver separately.
# ---------------------------------------------------------------

# Compute mean and standard deviation per driver
stats_per_driver = laps.groupby('Driver')['LapTimeSeconds'].agg(['mean', 'std']).rename(
    columns={'mean': 'driver_mean', 'std': 'driver_std'}
)

# Join these stats back to each lap row
laps = laps.merge(stats_per_driver, left_on='Driver', right_index=True)

# Compute z-score (how many SD a lap is above/below the driver's mean)
laps['z_score'] = (laps['LapTimeSeconds'] - laps['driver_mean']) / laps['driver_std']

# Keep only laps where z_score <= 2 (i.e., not more than 2 SD slower than the mean)
clean_laps = laps[laps['z_score'] <= 2].copy()

# Drop helper columns we no longer need
clean_laps = clean_laps.drop(columns=['driver_mean', 'driver_std', 'z_score'])

print(f"Total laps after cleaning: {len(clean_laps)}")

# ---------------------------------------------------------------
# 4. DESCRIPTIVE STATISTICS PER DRIVER (CLEAN LAPS ONLY)
# ---------------------------------------------------------------

# Helper function to compute the binned mode (nearest 0.5 seconds)
def binned_mode_half_second(values: pd.Series) -> float:
    # Divide by 0.5, floor to int, then multiply back by 0.5
    bins = np.floor(values / 0.5) * 0.5
    # value_counts().idxmax() returns the most frequent bin
    return bins.value_counts().idxmax()

# Group clean laps by driver and compute statistics
summary = []  # We will build a list of dictionaries and turn it into a DataFrame

for driver, group in clean_laps.groupby('Driver'):
    lap_times = group['LapTimeSeconds']

    mean_val = lap_times.mean()
    median_val = lap_times.median()
    mode_bin = binned_mode_half_second(lap_times)
    var_val = lap_times.var(ddof=1)          # Sample variance
    std_val = lap_times.std(ddof=1)          # Sample standard deviation
    range_val = lap_times.max() - lap_times.min()
    cv_val = (std_val / mean_val) * 100.0    # Coefficient of Variation in %

    summary.append({
        'Driver': driver,
        'Mean': mean_val,
        'Median': median_val,
        'ModeBin(0.5s)': mode_bin,
        'Variance': var_val,
        'StdDev': std_val,
        'Range': range_val,
        'CV(%)': cv_val,
    })

# Convert list of dictionaries to a DataFrame
stats_df = pd.DataFrame(summary)

# Round all numeric columns to 3 decimal places for cleaner display
numeric_cols = stats_df.columns.drop('Driver')
stats_df[numeric_cols] = stats_df[numeric_cols].round(3)

# ---------------------------------------------------------------
# Identify the driver with the lowest MEAN lap time and mark them
# with a checkmark in a new "Fastest" column.
# ---------------------------------------------------------------

fastest_driver = stats_df.loc[stats_df['Mean'].idxmin(), 'Driver']

# Create "Fastest" column: checkmark for the fastest, empty string for others
stats_df['Fastest'] = stats_df['Driver'].apply(
    lambda d: '✔' if d == fastest_driver else ''
)

print("Descriptive statistics for clean lap times (seconds):")
print(stats_df)

# Keep clean_laps and stats_df in memory for use in subsequent cells
clean_laps_section3 = clean_laps.copy()
stats_table_section3 = stats_df.copy()

In [ ]:
# ---------------------------------------------------------------
# SECTION 3 — CHART 1: Box Plot + Strip Plot of Clean Lap Times
# ---------------------------------------------------------------

import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.style.use('dark_background')  # Ensure dark style

# If the clean_laps DataFrame is not in memory (e.g. notebook re-run out of order),
# we reload the cleaned data logic by re-reading laps_data.csv and repeating
# the minimal cleaning steps used earlier.
try:
    clean_laps_plot = clean_laps_section3.copy()
except NameError:
    laps_path = './data/laps_data.csv'
    clean_laps_plot = pd.read_csv(laps_path)
    clean_laps_plot['LapNumber'] = pd.to_numeric(clean_laps_plot['LapNumber'], errors='coerce')

    if 'LapTimeSeconds' not in clean_laps_plot.columns and 'LapTime' in clean_laps_plot.columns:
        clean_laps_plot['LapTime'] = pd.to_timedelta(clean_laps_plot['LapTime'], errors='coerce')
        clean_laps_plot['LapTimeSeconds'] = clean_laps_plot['LapTime'].dt.total_seconds()

    clean_laps_plot = clean_laps_plot.dropna(subset=['LapTimeSeconds'])

    for col in ['PitInTime', 'PitOutTime']:
        if col in clean_laps_plot.columns:
            clean_laps_plot[col] = clean_laps_plot[col].replace(['', 'NaT', 'nan'], np.nan)
    if 'PitInTime' in clean_laps_plot.columns:
        clean_laps_plot = clean_laps_plot[clean_laps_plot['PitInTime'].isna()]
    if 'PitOutTime' in clean_laps_plot.columns:
        clean_laps_plot = clean_laps_plot[clean_laps_plot['PitOutTime'].isna()]
    clean_laps_plot = clean_laps_plot[clean_laps_plot['LapNumber'] != 1]

# Fixed driver colors
driver_colors = {
    'VER': '#E10600',
    'PER': '#0066CC',
    'SAI': '#FF8000',
    'RUS': '#C0C0C0',
}

# Create figure and axis
fig, ax = plt.subplots(figsize=(12, 7))

# Box plot of lap times per driver
sns.boxplot(
    data=clean_laps_plot,
    x='Driver',
    y='LapTimeSeconds',
    order=['VER', 'PER', 'SAI', 'RUS'],
    palette=driver_colors,
    ax=ax,
)

# Overlay individual lap times as a strip plot for extra detail
sns.stripplot(
    data=clean_laps_plot,
    x='Driver',
    y='LapTimeSeconds',
    order=['VER', 'PER', 'SAI', 'RUS'],
    palette=driver_colors,
    alpha=0.4,
    jitter=True,
    size=3,
    ax=ax,
)

ax.set_title('Lap Time Distribution — 2022 Belgian GP')
ax.set_xlabel('Driver')
ax.set_ylabel('Lap Time (seconds)')

# Add a simple watermark in the bottom-right corner of the axes
ax.text(
    1.0,
    0.0,
    'Data: FastF1 | 2022 Belgian GP',
    transform=ax.transAxes,
    ha='right',
    va='bottom',
    fontsize=9,
    color='white',
    alpha=0.7,
)

charts_dir = './charts'
os.makedirs(charts_dir, exist_ok=True)
boxplot_path = os.path.join(charts_dir, 'boxplot_laptimes.png')
plt.savefig(boxplot_path, dpi=150, bbox_inches='tight')

print(f"Saved box plot of lap times to {boxplot_path}")

plt.close(fig)

In [ ]:
# ---------------------------------------------------------------
# SECTION 3 — CHART 2: Rolling Standard Deviation (Window = 5 Laps)
# ---------------------------------------------------------------

import matplotlib.pyplot as plt
import os

plt.style.use('dark_background')  # Consistent dark style

# Reuse the clean_laps data prepared earlier if available
try:
    rolling_laps = clean_laps_section3.copy()
except NameError:
    laps_path = './data/laps_data.csv'
    rolling_laps = pd.read_csv(laps_path)
    rolling_laps['LapNumber'] = pd.to_numeric(rolling_laps['LapNumber'], errors='coerce')

    if 'LapTimeSeconds' not in rolling_laps.columns and 'LapTime' in rolling_laps.columns:
        rolling_laps['LapTime'] = pd.to_timedelta(rolling_laps['LapTime'], errors='coerce')
        rolling_laps['LapTimeSeconds'] = rolling_laps['LapTime'].dt.total_seconds()

    rolling_laps = rolling_laps.dropna(subset=['LapTimeSeconds'])

    for col in ['PitInTime', 'PitOutTime']:
        if col in rolling_laps.columns:
            rolling_laps[col] = rolling_laps[col].replace(['', 'NaT', 'nan'], np.nan)
    if 'PitInTime' in rolling_laps.columns:
        rolling_laps = rolling_laps[rolling_laps['PitInTime'].isna()]
    if 'PitOutTime' in rolling_laps.columns:
        rolling_laps = rolling_laps[rolling_laps['PitOutTime'].isna()]
    rolling_laps = rolling_laps[rolling_laps['LapNumber'] != 1]

# Sort by driver and lap number to ensure rolling window is in race order
rolling_laps = rolling_laps.sort_values(by=['Driver', 'LapNumber'])

# Compute rolling standard deviation of lap times (window = 5 laps) per driver
rolling_laps['RollingSD'] = (
    rolling_laps
    .groupby('Driver')['LapTimeSeconds']
    .rolling(window=5, min_periods=3)   # Require at least 3 laps to start computing SD
    .std()
    .reset_index(level=0, drop=True)
)

# Driver colors (same mapping as before)
driver_colors = {
    'VER': '#E10600',
    'PER': '#0066CC',
    'SAI': '#FF8000',
    'RUS': '#C0C0C0',
}

# Create the figure and axis
fig, ax = plt.subplots(figsize=(14, 6))

# Plot rolling SD lines for each driver
for driver_code, color in driver_colors.items():
    driver_data = rolling_laps[rolling_laps['Driver'] == driver_code]
    ax.plot(
        driver_data['LapNumber'],
        driver_data['RollingSD'],
        label=driver_code,
        color=color,
        linewidth=2.0,
    )

ax.set_title('Lap Time Consistency Over Race Distance (Rolling SD, window=5)')
ax.set_xlabel('Lap Number')
ax.set_ylabel('Rolling Standard Deviation (seconds)')

ax.grid(alpha=0.2, linestyle='--')
ax.legend(title='Driver', loc='upper right')

# Optional watermark
ax.text(
    1.0,
    0.0,
    'Data: FastF1 | 2022 Belgian GP',
    transform=ax.transAxes,
    ha='right',
    va='bottom',
    fontsize=9,
    color='white',
    alpha=0.7,
)

charts_dir = './charts'
os.makedirs(charts_dir, exist_ok=True)
rolling_sd_path = os.path.join(charts_dir, 'rolling_sd.png')
plt.savefig(rolling_sd_path, dpi=150, bbox_inches='tight')

print(f"Saved rolling standard deviation chart to {rolling_sd_path}")

plt.close(fig)

<a id="section-3"></a>
## Section 3 — Descriptive Statistics

The descriptive statistics table shows that Verstappen not only has the **lowest mean lap time**, but also one of the **lowest standard deviations**, meaning his laps were both faster *and* more consistent than the other three drivers. The box plot and overlaid lap-time dots confirm this visually: VER's box is shifted lower (faster) and is relatively tight, while the other drivers have slower and more spread-out lap distributions. The rolling standard deviation chart shows that VER's lap-time variability stays low across the race distance, which is the first clear statistical proof that his charge from P14 was built on a dominant and repeatable pace advantage, not just a few lucky laps.

In [ ]:
# ---------------------------------------------------------------
# SECTION 4 — Normal Distribution & Z-Score (Clean Lap Times)
# ---------------------------------------------------------------

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats  # We use stats.norm for normal fitting and CDF
import os

plt.style.use('dark_background')  # Dark background for this section

# ---------------------------------------------------------------
# Reuse the clean lap data from Section 3 if it exists in memory.
# If not (e.g. user ran cells out of order), rebuild it using the
# same cleaning rules as in Section 3.
# ---------------------------------------------------------------

try:
    clean_laps_4 = clean_laps_section3.copy()
except NameError:
    # Fallback: re-load laps and apply Section 3 cleaning logic
    import pandas as pd

    laps_path = './data/laps_data.csv'
    laps = pd.read_csv(laps_path)
    laps['LapNumber'] = pd.to_numeric(laps['LapNumber'], errors='coerce')

    if 'LapTimeSeconds' not in laps.columns and 'LapTime' in laps.columns:
        laps['LapTime'] = pd.to_timedelta(laps['LapTime'], errors='coerce')
        laps['LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()

    laps = laps.dropna(subset=['LapTimeSeconds'])

    # Standardize pit time columns
    for col in ['PitInTime', 'PitOutTime']:
        if col in laps.columns:
            laps[col] = laps[col].replace(['', 'NaT', 'nan'], np.nan)

    # Remove pit in / pit out laps
    if 'PitInTime' in laps.columns:
        laps = laps[laps['PitInTime'].isna()]
    if 'PitOutTime' in laps.columns:
        laps = laps[laps['PitOutTime'].isna()]

    # Remove first lap for all drivers
    laps = laps[laps['LapNumber'] != 1]

    # Remove laps > 2 SD slower than each driver's mean
    stats_per_driver = laps.groupby('Driver')['LapTimeSeconds'].agg(['mean', 'std']).rename(
        columns={'mean': 'driver_mean', 'std': 'driver_std'}
    )
    laps = laps.merge(stats_per_driver, left_on='Driver', right_index=True)
    laps['z_score'] = (laps['LapTimeSeconds'] - laps['driver_mean']) / laps['driver_std']
    clean_laps_4 = laps[laps['z_score'] <= 2].copy()
    clean_laps_4 = clean_laps_4.drop(columns=['driver_mean', 'driver_std', 'z_score'])

# ---------------------------------------------------------------
# CHART 1 — VER Histogram with Normal Fit
# ---------------------------------------------------------------

# Filter only Verstappen's clean laps
ver_laps = clean_laps_4[clean_laps_4['Driver'] == 'VER']
ver_times = ver_laps['LapTimeSeconds'].values

# Fit a normal distribution to Verstappen's lap times.
# norm.fit returns the maximum-likelihood estimates (mu, sigma).
mu_ver, sigma_ver = stats.norm.fit(ver_times)

# Create a figure and axis for the histogram + fitted curve
fig, ax = plt.subplots(figsize=(12, 6))

# Plot histogram of lap times with 15 bins.
# density=False means the y-axis is "count" (frequency), not probability.
counts, bin_edges, _ = ax.hist(
    ver_times,
    bins=15,
    density=False,
    alpha=0.6,
    color='#E10600',  # Red-ish color for VER
    edgecolor='white',
)

# Build x-values spanning the full range of Verstappen's lap times
x = np.linspace(ver_times.min(), ver_times.max(), 300)

# To overlay the normal curve on a frequency histogram, we scale the
# probability density function (PDF) by (number of samples * bin width).
bin_width = bin_edges[1] - bin_edges[0]
scale = len(ver_times) * bin_width
pdf = stats.norm.pdf(x, mu_ver, sigma_ver) * scale

ax.plot(x, pdf, 'w-', linewidth=2.0, label='Normal fit')

# Add a vertical dashed line at Verstappen's mean lap time
ax.axvline(mu_ver, color='white', linestyle='--', linewidth=1.5, label='Mean')

# Shade the area within one standard deviation of the mean
x_shade = np.linspace(mu_ver - sigma_ver, mu_ver + sigma_ver, 200)
pdf_shade = stats.norm.pdf(x_shade, mu_ver, sigma_ver) * scale
ax.fill_between(x_shade, pdf_shade, color='white', alpha=0.15)

# Labels and title
ax.set_title('Verstappen Lap Time Distribution — Normal Distribution Fit')
ax.set_xlabel('Lap Time (seconds)')
ax.set_ylabel('Frequency')

ax.legend(loc='upper right')

# Watermark
ax.text(
    1.0,
    0.0,
    'Data: FastF1 | 2022 Belgian GP',
    transform=ax.transAxes,
    ha='right',
    va='bottom',
    fontsize=9,
    color='white',
    alpha=0.7,
)

charts_dir = './charts'
os.makedirs(charts_dir, exist_ok=True)
ver_hist_path = os.path.join(charts_dir, 'ver_normal_fit.png')
plt.savefig(ver_hist_path, dpi=150, bbox_inches='tight')

print(f"Saved Verstappen normal-fit histogram to {ver_hist_path}")

plt.close(fig)

# ---------------------------------------------------------------
# Z-SCORE CALCULATION (Based on Clean Lap Means for 4 Drivers)
# ---------------------------------------------------------------

# Compute mean lap time for each driver using clean laps
means_by_driver = clean_laps_4.groupby('Driver')['LapTimeSeconds'].mean()

# Keep only the four drivers of interest (VER, PER, SAI, RUS)
selected_drivers = ['VER', 'PER', 'SAI', 'RUS']
means_by_driver = means_by_driver.loc[selected_drivers]

# Field mean = average of the four drivers' mean lap times
field_mean = means_by_driver.mean()

# Field SD = standard deviation of the four drivers' mean lap times
field_sd = means_by_driver.std(ddof=1)

# Verstappen's mean lap time
ver_mean = means_by_driver['VER']

# Z-score: how many standard deviations Verstappen's mean is from the field mean
z_ver = (ver_mean - field_mean) / field_sd

# Percentile: where Verstappen sits in the field distribution
# Because lower lap times are better, a negative Z-score means
# Verstappen is faster than average; we use the CDF of the normal
# distribution to compute the percentile.
percentile_ver = stats.norm.cdf(z_ver) * 100.0

print(f"Field Mean Lap Time: {field_mean:.3f} seconds")
print(f"Field Standard Deviation: {field_sd:.3f} seconds")
print(f"Verstappen Z-Score: {z_ver:.2f}")
print(
    f"Interpretation: Verstappen was {abs(z_ver):.2f} standard deviations "
    "faster than the field average. This is a statistically significant performance gap."
)
print(f"Verstappen performed faster than {percentile_ver:.1f}% of the field distribution.")

# Keep these in memory for later use in the report-style markdown if needed
z_score_ver = float(z_ver)
field_mean_laptime = float(field_mean)
field_sd_laptime = float(field_sd)
ver_mean_laptime = float(ver_mean)
ver_percentile = float(percentile_ver)

In [ ]:
# ---------------------------------------------------------------
# SECTION 4 — CHART 2: All Drivers Normal Curves Compared
# ---------------------------------------------------------------

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import os

plt.style.use('dark_background')  # Dark background

# Reuse clean_laps_4 from the previous cell (already cleaned)
all_clean = clean_laps_4.copy()

# Restrict to the four drivers we care about
drivers_of_interest = ['VER', 'PER', 'SAI', 'RUS']
all_clean = all_clean[all_clean['Driver'].isin(drivers_of_interest)]

# Driver colors
driver_colors = {
    'VER': '#E10600',
    'PER': '#0066CC',
    'SAI': '#FF8000',
    'RUS': '#C0C0C0',
}

# Compute overall min and max lap time for setting a common x-axis range
overall_min = all_clean['LapTimeSeconds'].min()
overall_max = all_clean['LapTimeSeconds'].max()

x = np.linspace(overall_min, overall_max, 400)

fig, ax = plt.subplots(figsize=(14, 6))

for driver_code in drivers_of_interest:
    driver_times = all_clean[all_clean['Driver'] == driver_code]['LapTimeSeconds'].values

    # Fit a normal distribution (mu, sigma) for this driver's lap times
    mu, sigma = stats.norm.fit(driver_times)

    # Compute the probability density function (PDF) values at x
    pdf = stats.norm.pdf(x, mu, sigma)

    ax.plot(
        x,
        pdf,
        color=driver_colors[driver_code],
        linewidth=2.0,
        label=f"{driver_code}",
    )

    # Vertical dashed line at this driver's mean lap time
    ax.axvline(
        mu,
        color=driver_colors[driver_code],
        linestyle='--',
        linewidth=1.2,
        alpha=0.8,
    )

ax.set_title('Lap Time Normal Distribution — All Drivers Compared')
ax.set_xlabel('Lap Time (seconds)')
ax.set_ylabel('Probability Density')

ax.legend(title='Driver', loc='upper right')
ax.grid(alpha=0.2, linestyle='--')

# Simple watermark
ax.text(
    1.0,
    0.0,
    'Data: FastF1 | 2022 Belgian GP',
    transform=ax.transAxes,
    ha='right',
    va='bottom',
    fontsize=9,
    color='white',
    alpha=0.7,
)

charts_dir = './charts'
os.makedirs(charts_dir, exist_ok=True)
all_normals_path = os.path.join(charts_dir, 'all_drivers_normal.png')
plt.savefig(all_normals_path, dpi=150, bbox_inches='tight')

print(f"Saved all-drivers normal distribution chart to {all_normals_path}")

plt.close(fig)

<a id="section-4"></a>
## Section 4 — Normal Distribution & Z-Score

In this section we treated Verstappen’s clean lap times as samples from a (roughly) normal distribution, fit a bell curve to them, and compared his average pace to the “field” made up of PER, SAI, and RUS. The histogram plus fitted normal curve show that most of Verstappen’s laps are tightly clustered around a very low (fast) mean, with relatively small spread — the shaded band within one standard deviation covers a narrow slice of the x-axis.

The Z-score we computed tells us how many standard deviations Verstappen’s mean lap time sits **below** the field’s average mean lap time: a more negative Z-score means a bigger pace advantage. If your printed Verstappen Z-score is around **-1.5 or lower**, you can interpret it like this:

> A Z-score of X means that in a random sample from the field's distribution, only X% of laps would naturally be as fast as Verstappen's average. This is not normal variation — this is a statistically dominant performance.

The second chart, with normal curves for all four drivers, makes the story visual: Verstappen’s curve is shifted left (faster) and often taller around its peak, while the others are shifted to the right with broader shapes. Together, the Z-score and the normal-curve comparison show that his P14-to-P1 drive wasn’t just aggressive racecraft — it was backed by a lap-time distribution that made anything other than victory statistically unlikely.

In [ ]:
# ---------------------------------------------------------------
# SECTION 5 — Tire Degradation Linear Regression
# Syllabus: Unit V — Regression and straight-line fitting
# ---------------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import os

plt.style.use('dark_background')  # Dark background style for consistency

# ---------------------------------------------------------------
# 1) LOAD LAPS AND REBUILD "CLEAN" LAPS (same rules as Section 3)
# ---------------------------------------------------------------

laps_path = './data/laps_data.csv'          # Laps exported in Section 1
laps = pd.read_csv(laps_path)               # Read laps into a DataFrame

# Ensure key columns are numeric where appropriate
laps['LapNumber'] = pd.to_numeric(laps['LapNumber'], errors='coerce')

# Ensure we have lap time in seconds (float)
if 'LapTimeSeconds' not in laps.columns and 'LapTime' in laps.columns:
    # If LapTime is present as a string, parse to timedelta then convert to seconds
    laps['LapTime'] = pd.to_timedelta(laps['LapTime'], errors='coerce')
    laps['LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()

# Drop rows where lap time is missing
laps = laps.dropna(subset=['LapTimeSeconds'])

# Remove pit-in / pit-out laps (these are not representative "clean" racing laps)
for col in ['PitInTime', 'PitOutTime']:
    if col in laps.columns:
        # Replace common string placeholders with real missing values
        laps[col] = laps[col].replace(['', 'NaT', 'nan'], np.nan)

if 'PitInTime' in laps.columns:
    laps = laps[laps['PitInTime'].isna()]
if 'PitOutTime' in laps.columns:
    laps = laps[laps['PitOutTime'].isna()]

# Remove the first lap (lap 1), which includes the start and is not a "steady-state" lap
laps = laps[laps['LapNumber'] != 1]

# Remove laps more than 2 standard deviations slower than each driver's mean
per_driver = laps.groupby('Driver')['LapTimeSeconds'].agg(['mean', 'std']).rename(
    columns={'mean': 'driver_mean', 'std': 'driver_std'}
)

laps = laps.merge(per_driver, left_on='Driver', right_index=True)

# z_score > 2 means the lap is more than 2 SD slower than that driver's average
laps['z_score'] = (laps['LapTimeSeconds'] - laps['driver_mean']) / laps['driver_std']

clean_laps = laps[laps['z_score'] <= 2].copy()
clean_laps = clean_laps.drop(columns=['driver_mean', 'driver_std', 'z_score'])

# ---------------------------------------------------------------
# 2) SET UP REGRESSION DATA (TyreLife vs LapTimeSeconds)
# ---------------------------------------------------------------

drivers_of_interest = ['VER', 'PER', 'SAI', 'RUS']  # Exactly these four drivers
clean_laps = clean_laps[clean_laps['Driver'].isin(drivers_of_interest)]

# TyreLife is our X variable: number of laps on the current tyre set
# LapTimeSeconds is our Y variable: lap time in seconds (float)

# Convert TyreLife to numeric and drop missing X/Y values
clean_laps['TyreLife'] = pd.to_numeric(clean_laps.get('TyreLife'), errors='coerce')
clean_laps['LapTimeSeconds'] = pd.to_numeric(clean_laps['LapTimeSeconds'], errors='coerce')

regression_data = clean_laps.dropna(subset=['TyreLife', 'LapTimeSeconds']).copy()

print(f"Clean laps available for regression: {len(regression_data)}")

# ---------------------------------------------------------------
# 3) RUN LINEAR REGRESSION PER DRIVER
# We use scipy.stats.linregress which fits y = slope*x + intercept.
# ---------------------------------------------------------------

results = []  # Store regression outputs for a summary table

for driver in drivers_of_interest:
    df_driver = regression_data[regression_data['Driver'] == driver]

    # Extract X (tyre age) and Y (lap time)
    x = df_driver['TyreLife'].values
    y = df_driver['LapTimeSeconds'].values

    # Guard: if there are too few points, regression is not meaningful
    if len(x) < 3:
        results.append({
            'Driver': driver,
            'Slope (s/lap)': np.nan,
            'Intercept': np.nan,
            'R²': np.nan,
            'p-value': np.nan,
            'Interpretation': 'Not enough data',
        })
        continue

    # Run the regression
    lr = stats.linregress(x, y)

    slope = lr.slope
    intercept = lr.intercept
    r_value = lr.rvalue
    p_value = lr.pvalue
    std_err = lr.stderr

    r_squared = r_value ** 2

    # Interpretation rules based on slope size
    if slope < 0.05:
        interpretation = 'Very low degradation'
    elif 0.05 <= slope < 0.10:
        interpretation = 'Low degradation'
    elif 0.10 <= slope < 0.15:
        interpretation = 'Moderate degradation'
    else:
        interpretation = 'High degradation'

    results.append({
        'Driver': driver,
        'Slope (s/lap)': slope,
        'Intercept': intercept,
        'R²': r_squared,
        'p-value': p_value,
        'StdErr': std_err,
        'Interpretation': interpretation,
    })

# Build a clean summary table (drop StdErr from display, but we keep it in results list if you need it)
regression_table = pd.DataFrame(results)

# Round numeric columns for readability
for col in ['Slope (s/lap)', 'Intercept', 'R²', 'p-value']:
    if col in regression_table.columns:
        regression_table[col] = regression_table[col].round(4)

# Display the summary table in a neat format
print("\nRegression Summary (TyreLife vs Lap Time):")
print(regression_table[['Driver', 'Slope (s/lap)', 'Intercept', 'R²', 'p-value', 'Interpretation']])

# ---------------------------------------------------------------
# 4) CHART — Scatter + Regression Lines (All Drivers)
# ---------------------------------------------------------------

driver_colors = {
    'VER': '#E10600',
    'PER': '#0066CC',
    'SAI': '#FF8000',
    'RUS': '#C0C0C0',
}

fig, ax = plt.subplots(figsize=(14, 7))

# Plot scatter points and regression line for each driver
for row in results:
    driver = row['Driver']
    df_driver = regression_data[regression_data['Driver'] == driver]

    # Scatter plot of raw data points
    ax.scatter(
        df_driver['TyreLife'],
        df_driver['LapTimeSeconds'],
        color=driver_colors[driver],
        alpha=0.35,
        s=30,
        label=driver,
    )

    # If slope is NaN (not enough data), skip line plotting
    if pd.isna(row.get('Slope (s/lap)')):
        continue

    slope = float(row['Slope (s/lap)'])
    intercept = float(row['Intercept'])

    # Build a line across the observed tyre life range
    x_min = df_driver['TyreLife'].min()
    x_max = df_driver['TyreLife'].max()
    x_line = np.array([x_min, x_max])
    y_line = slope * x_line + intercept

    ax.plot(
        x_line,
        y_line,
        color=driver_colors[driver],
        linewidth=2,
    )

    # Label the regression line at the right end with slope text
    ax.text(
        x_max,
        y_line[-1],
        f"{driver}: {slope:+.2f}s/lap",
        color=driver_colors[driver],
        fontsize=10,
        ha='left',
        va='center',
    )

ax.set_title('Tire Degradation Rate — Linear Regression (2022 Belgian GP)')
ax.set_xlabel('Tire Age (laps)')
ax.set_ylabel('Lap Time (seconds)')

ax.grid(alpha=0.2, linestyle='--')

# Watermark bottom-right
ax.text(
    1.0,
    0.0,
    'Data: FastF1 | 2022 Belgian GP',
    transform=ax.transAxes,
    ha='right',
    va='bottom',
    fontsize=9,
    color='white',
    alpha=0.7,
)

# Save chart
charts_dir = './charts'
os.makedirs(charts_dir, exist_ok=True)
reg_path = os.path.join(charts_dir, 'tire_regression.png')
plt.savefig(reg_path, dpi=150, bbox_inches='tight')

print(f"Saved tire degradation regression chart to {reg_path}")

plt.close(fig)

# Keep the regression table in memory for later sections
regression_table_section5 = regression_table.copy()

<a id="section-5"></a>
## Section 5 — Tire Degradation Regression

In the regression model, the **slope** is the key number: it estimates how many **seconds slower** a driver becomes for every additional lap on the same tires (for example, **+0.10 s/lap** means the car gets about one tenth of a second slower each lap as the tires age). A **flatter (smaller) slope** means the driver’s pace is less affected by tire wear — they can sustain performance deeper into a stint.

Look at the summary table and identify the **lowest slope**: that driver had the best tire management (or at least the least measurable degradation in this race). The VER vs PER comparison is especially important because they had the same car; if VER’s slope is lower than PER’s, it suggests Verstappen could push harder for longer without paying the same tire-wear penalty, which helps explain how his climb through the field was sustainable across the race distance.

In [ ]:
# ---------------------------------------------------------------
# SECTION 6 — Pearson Correlation (TyreLife vs Lap Time)
# Syllabus: Unit V — Covariance and Pearson's Correlation
# ---------------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import os

plt.style.use('dark_background')  # Dark background style

# ---------------------------------------------------------------
# DATA: Use the same clean laps + (TyreLife, LapTimeSeconds) setup
# as Section 5. If the Section 5 regression_data DataFrame exists,
# reuse it. Otherwise, rebuild it from laps_data.csv with the same
# cleaning rules.
# ---------------------------------------------------------------

try:
    corr_data = regression_data.copy()  # Created in Section 5 after cleaning + dropping nulls
except NameError:
    # Fallback: rebuild the regression/correlation dataset from scratch
    laps_path = './data/laps_data.csv'
    laps = pd.read_csv(laps_path)

    # Ensure LapNumber is numeric
    laps['LapNumber'] = pd.to_numeric(laps['LapNumber'], errors='coerce')

    # Ensure lap time is in seconds (float)
    if 'LapTimeSeconds' not in laps.columns and 'LapTime' in laps.columns:
        laps['LapTime'] = pd.to_timedelta(laps['LapTime'], errors='coerce')
        laps['LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()

    # Drop rows missing lap time
    laps = laps.dropna(subset=['LapTimeSeconds'])

    # Remove pit-in / pit-out laps
    for col in ['PitInTime', 'PitOutTime']:
        if col in laps.columns:
            laps[col] = laps[col].replace(['', 'NaT', 'nan'], np.nan)

    if 'PitInTime' in laps.columns:
        laps = laps[laps['PitInTime'].isna()]
    if 'PitOutTime' in laps.columns:
        laps = laps[laps['PitOutTime'].isna()]

    # Remove lap 1
    laps = laps[laps['LapNumber'] != 1]

    # Remove laps > 2 SD slower than each driver's mean lap time
    per_driver = laps.groupby('Driver')['LapTimeSeconds'].agg(['mean', 'std']).rename(
        columns={'mean': 'driver_mean', 'std': 'driver_std'}
    )
    laps = laps.merge(per_driver, left_on='Driver', right_index=True)
    laps['z_score'] = (laps['LapTimeSeconds'] - laps['driver_mean']) / laps['driver_std']
    clean_laps = laps[laps['z_score'] <= 2].copy()
    clean_laps = clean_laps.drop(columns=['driver_mean', 'driver_std', 'z_score'])

    # Keep only the four drivers of interest
    drivers_of_interest = ['VER', 'PER', 'SAI', 'RUS']
    clean_laps = clean_laps[clean_laps['Driver'].isin(drivers_of_interest)]

    # Convert TyreLife to numeric and drop missing X/Y values
    clean_laps['TyreLife'] = pd.to_numeric(clean_laps.get('TyreLife'), errors='coerce')
    clean_laps['LapTimeSeconds'] = pd.to_numeric(clean_laps['LapTimeSeconds'], errors='coerce')

    corr_data = clean_laps.dropna(subset=['TyreLife', 'LapTimeSeconds']).copy()

# Ensure we only use the four target drivers (no others)
drivers_of_interest = ['VER', 'PER', 'SAI', 'RUS']
corr_data = corr_data[corr_data['Driver'].isin(drivers_of_interest)]

print(f"Rows available for correlation: {len(corr_data)}")

# ---------------------------------------------------------------
# CALCULATIONS: covariance, Pearson r, p-value, and interpretation
# ---------------------------------------------------------------

# Map r strength based on absolute value (strength ignores direction)
def interpret_strength(r_value: float) -> str:
    r_abs = abs(r_value)
    if 0.00 <= r_abs <= 0.19:
        return 'Negligible correlation'
    if 0.20 <= r_abs <= 0.39:
        return 'Weak correlation'
    if 0.40 <= r_abs <= 0.59:
        return 'Moderate correlation'
    if 0.60 <= r_abs <= 0.79:
        return 'Strong correlation'
    return 'Very strong correlation'

summary_rows = []

for driver in drivers_of_interest:
    df_driver = corr_data[corr_data['Driver'] == driver]

    tyre_age = df_driver['TyreLife'].to_numpy(dtype=float)        # X values
    lap_time = df_driver['LapTimeSeconds'].to_numpy(dtype=float)  # Y values

    # If a driver has too few points, correlation is not reliable
    if len(tyre_age) < 3:
        summary_rows.append({
            'Driver': driver,
            'Covariance': np.nan,
            'Pearson r': np.nan,
            'p-value': np.nan,
            'Strength': 'Not enough data',
            'Significant?': 'No',
        })
        continue

    # Covariance matrix: [ [cov(x,x), cov(x,y)], [cov(y,x), cov(y,y)] ]
    cov_matrix = np.cov(tyre_age, lap_time)  # Uses sample covariance (N-1) by default
    covariance_xy = cov_matrix[0, 1]         # Covariance between TyreLife and LapTimeSeconds

    # Pearson correlation coefficient and two-sided p-value
    r, p = stats.pearsonr(tyre_age, lap_time)

    strength = interpret_strength(r)
    significant = 'Yes' if p < 0.05 else 'No'

    summary_rows.append({
        'Driver': driver,
        'Covariance': covariance_xy,
        'Pearson r': r,
        'p-value': p,
        'Strength': strength,
        'Significant?': significant,
    })

summary_df = pd.DataFrame(summary_rows)

# Round for neat display
summary_df['Covariance'] = summary_df['Covariance'].round(4)
summary_df['Pearson r'] = summary_df['Pearson r'].round(3)
summary_df['p-value'] = summary_df['p-value'].round(4)

print("\nPearson Correlation Summary (TyreLife vs Lap Time):")
print(summary_df[['Driver', 'Covariance', 'Pearson r', 'p-value', 'Strength', 'Significant?']])

# ---------------------------------------------------------------
# CHART — Horizontal bar chart of Pearson r values
# ---------------------------------------------------------------

driver_colors = {
    'VER': '#E10600',
    'PER': '#0066CC',
    'SAI': '#FF8000',
    'RUS': '#C0C0C0',
}

# Prepare plotting data in the requested driver order
plot_df = summary_df.set_index('Driver').loc[drivers_of_interest].reset_index()

fig, ax = plt.subplots(figsize=(10, 5))

# Horizontal bars: y = driver codes, x = Pearson r values
bars = ax.barh(
    plot_df['Driver'],
    plot_df['Pearson r'],
    color=[driver_colors[d] for d in plot_df['Driver']],
    alpha=0.9,
)

# Vertical dashed line at r = 0 to show positive vs negative correlation
ax.axvline(0, color='white', linestyle='--', linewidth=1.2, alpha=0.9)

# Label each bar with its r value (e.g., "r = 0.42")
for bar, r_val in zip(bars, plot_df['Pearson r']):
    # bar.get_width() is the r value, bar.get_y() is the y position
    x_text = bar.get_width()
    y_text = bar.get_y() + bar.get_height() / 2

    # Offset label slightly to the right for positive bars, left for negative
    offset = 0.02 if x_text >= 0 else -0.02
    ha = 'left' if x_text >= 0 else 'right'

    ax.text(
        x_text + offset,
        y_text,
        f"r = {r_val:.2f}",
        va='center',
        ha=ha,
        color='white',
        fontsize=10,
    )

ax.set_title('Pearson Correlation: Tire Age vs Lap Time')
ax.set_xlabel("Pearson r (correlation coefficient)")

# Add the requested explanatory annotation
ax.text(
    0.99,
    0.05,
    "→ Higher r = tires affected pace more",
    transform=ax.transAxes,
    ha='right',
    va='bottom',
    fontsize=10,
    color='white',
    alpha=0.85,
)

ax.grid(alpha=0.2, linestyle='--', axis='x')

# Save chart
charts_dir = './charts'
os.makedirs(charts_dir, exist_ok=True)
pearson_path = os.path.join(charts_dir, 'pearson_correlation.png')
plt.savefig(pearson_path, dpi=150, bbox_inches='tight')

print(f"Saved Pearson correlation chart to {pearson_path}")

plt.close(fig)

# Keep table in memory for later sections
pearson_table_section6 = summary_df.copy()

<a id="section-6"></a>
## Section 6 — Pearson Correlation

Pearson’s correlation coefficient **r** measures how strongly tire age (TyreLife) and lap time move together. A **higher positive r** means that as tires get older, lap times tend to get slower in a more predictable way — in other words, tire condition is strongly controlling pace. A **lower r** means lap time is less tightly linked to tire age, which often suggests the driver/car can keep pushing without pace dropping in a simple “older tires = slower lap” pattern.

In this race story, the driver with the **lowest r (closest to 0)** had pace that was least controlled by tire age, which can be a sign of strong tire management or a car/driver combination that stays stable across a stint. If Verstappen’s r is among the lowest, it supports the idea that he could sustain his charge through the field because his pace didn’t fade as strongly with tire age — he could keep producing fast, consistent laps even as the stint progressed.

In [ ]:
# ---------------------------------------------------------------
# SECTION 7 — Telemetry Analysis (Fastest Lap Telemetry)
# Visual highlight section
# ---------------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

plt.style.use('dark_background')  # Dark background for telemetry charts

# ---------------------------------------------------------------
# Configuration: drivers, colors, and key Spa track landmarks
# ---------------------------------------------------------------

drivers = ['VER', 'PER', 'SAI', 'RUS']

driver_colors = {
    'VER': '#E10600',
    'PER': '#0066CC',
    'SAI': '#FF8000',
    'RUS': '#C0C0C0',
}

# Approximate distances (in metres) along the lap for key Spa sections
spa_markers = [
    (1000, 'Eau Rouge / Raidillon'),
    (2000, 'Kemmel Straight'),
    (4200, 'Les Combes'),
    (6500, 'Bus Stop Chicane'),
]

data_dir = './data'
charts_dir = './charts'
os.makedirs(charts_dir, exist_ok=True)

# ---------------------------------------------------------------
# Helper: add vertical dashed lines + labels for Spa markers
# We place labels at the top of the axes and rotate them 90 degrees.
# ---------------------------------------------------------------

def add_spa_annotations(ax):
    for x_m, label in spa_markers:
        ax.axvline(x_m, color='white', linestyle='--', alpha=0.4, linewidth=1.0)
        ax.text(
            x_m,
            0.98,
            label,
            transform=ax.get_xaxis_transform(),  # x in data coords, y in axes coords
            rotation=90,
            va='top',
            ha='right',
            fontsize=9,
            color='white',
            alpha=0.85,
        )

# ---------------------------------------------------------------
# 1) LOAD TELEMETRY FILES (skip missing files with a warning)
# ---------------------------------------------------------------

telemetry_by_driver = {}

for d in drivers:
    path = os.path.join(data_dir, f'telemetry_{d}.csv')

    if not os.path.exists(path):
        print(f"WARNING: Missing telemetry file: {path}. Skipping {d}.")
        continue

    df = pd.read_csv(path)

    # Ensure required columns exist
    required_cols = ['Distance']
    for col in required_cols:
        if col not in df.columns:
            print(f"WARNING: {path} is missing column '{col}'. Skipping {d}.")
            df = None
            break

    if df is None:
        continue

    # Convert Distance to numeric and drop bad rows
    df['Distance'] = pd.to_numeric(df['Distance'], errors='coerce')
    df = df.dropna(subset=['Distance']).sort_values('Distance')

    telemetry_by_driver[d] = df

if len(telemetry_by_driver) == 0:
    raise RuntimeError("No telemetry files were found/loaded. Run Section 1 first to export telemetry CSVs.")

# ---------------------------------------------------------------
# 2) ALIGN TELEMETRY BY DISTANCE
# ---------------------------------------------------------------
# Different drivers may have telemetry sampled at slightly different
# distance points. To compare traces fairly, we build a shared distance
# grid over the OVERLAPPING distance range and interpolate each driver.
# ---------------------------------------------------------------

# Compute the overlapping distance range across available drivers
min_common = max(df['Distance'].min() for df in telemetry_by_driver.values())
max_common = min(df['Distance'].max() for df in telemetry_by_driver.values())

# If overlap is too small, interpolation becomes meaningless
if max_common <= min_common:
    raise RuntimeError("Telemetry distance ranges do not overlap. Cannot align traces.")

# Build a shared grid (1000 points gives a smooth comparison)
distance_grid = np.linspace(min_common, max_common, 1000)

# Interpolate Speed onto the shared distance grid for each driver
speed_aligned = {}

for d, df in telemetry_by_driver.items():
    if 'Speed' not in df.columns:
        print(f"WARNING: telemetry_{d}.csv missing 'Speed'. Skipping speed trace for {d}.")
        continue

    # Ensure Speed is numeric
    speed = pd.to_numeric(df['Speed'], errors='coerce').to_numpy(dtype=float)
    dist = df['Distance'].to_numpy(dtype=float)

    # Drop rows where speed is missing before interpolation
    valid = ~np.isnan(speed)
    dist_valid = dist[valid]
    speed_valid = speed[valid]

    if len(dist_valid) < 2:
        print(f"WARNING: Not enough speed points for {d}. Skipping.")
        continue

    # np.interp performs 1D linear interpolation
    speed_aligned[d] = np.interp(distance_grid, dist_valid, speed_valid)

# ---------------------------------------------------------------
# CHART 1 — Speed Trace Comparison
# ---------------------------------------------------------------

fig, ax = plt.subplots(figsize=(16, 6))

for d in drivers:
    if d not in speed_aligned:
        continue

    ax.plot(
        distance_grid,
        speed_aligned[d],
        color=driver_colors[d],
        linewidth=1.5,
        alpha=0.9,
        label=d,
    )

add_spa_annotations(ax)

ax.set_title('Speed Trace Comparison — Fastest Lap (2022 Belgian GP)')
ax.set_xlabel('Distance (m)')
ax.set_ylabel('Speed (km/h)')

ax.grid(alpha=0.2, linestyle='--')
ax.legend(title='Driver', loc='upper right')

# Watermark
ax.text(
    1.0,
    0.0,
    'Data: FastF1 | 2022 Belgian GP',
    transform=ax.transAxes,
    ha='right',
    va='bottom',
    fontsize=9,
    color='white',
    alpha=0.7,
)

speed_path = os.path.join(charts_dir, 'telemetry_speed.png')
plt.savefig(speed_path, dpi=150, bbox_inches='tight')
print(f"Saved telemetry speed chart to {speed_path}")

plt.close(fig)

<a id="section-7"></a>
## Section 7 — Telemetry Analysis

The Kemmel Straight section (around the **~2000m** marker) is where top speed matters most: the driver whose speed trace sits highest there has the best straight-line performance on the fastest lap. Higher top speed usually means a stronger ability to **attack and overtake**, because it increases closing speed and makes it harder for the car ahead to defend. If VER’s line is consistently on top through Kemmel, it visually supports why passing through the field was realistic — he had the straight-line pace to complete moves, not just follow closely.

In [ ]:
# ---------------------------------------------------------------
# SECTION 7 — CHART 2+3: Throttle and Brake (stacked subplots)
# ---------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

plt.style.use('dark_background')

# Reuse telemetry_by_driver and distance_grid from the speed chart cell if available.
# If not available (cell run out of order), reload telemetry and rebuild alignment.
try:
    _ = telemetry_by_driver
    _ = distance_grid
except NameError:
    drivers = ['VER', 'PER', 'SAI', 'RUS']

    driver_colors = {
        'VER': '#E10600',
        'PER': '#0066CC',
        'SAI': '#FF8000',
        'RUS': '#C0C0C0',
    }

    spa_markers = [
        (1000, 'Eau Rouge / Raidillon'),
        (2000, 'Kemmel Straight'),
        (4200, 'Les Combes'),
        (6500, 'Bus Stop Chicane'),
    ]

    def add_spa_annotations(ax):
        for x_m, label in spa_markers:
            ax.axvline(x_m, color='white', linestyle='--', alpha=0.4, linewidth=1.0)
            ax.text(
                x_m,
                0.98,
                label,
                transform=ax.get_xaxis_transform(),
                rotation=90,
                va='top',
                ha='right',
                fontsize=9,
                color='white',
                alpha=0.85,
            )

    data_dir = './data'
    telemetry_by_driver = {}

    for d in drivers:
        path = os.path.join(data_dir, f'telemetry_{d}.csv')
        if not os.path.exists(path):
            print(f"WARNING: Missing telemetry file: {path}. Skipping {d}.")
            continue

        df = pd.read_csv(path)
        if 'Distance' not in df.columns:
            print(f"WARNING: {path} is missing 'Distance'. Skipping {d}.")
            continue

        df['Distance'] = pd.to_numeric(df['Distance'], errors='coerce')
        df = df.dropna(subset=['Distance']).sort_values('Distance')
        telemetry_by_driver[d] = df

    if len(telemetry_by_driver) == 0:
        raise RuntimeError("No telemetry files were found/loaded.")

    min_common = max(df['Distance'].min() for df in telemetry_by_driver.values())
    max_common = min(df['Distance'].max() for df in telemetry_by_driver.values())

    if max_common <= min_common:
        raise RuntimeError("Telemetry distance ranges do not overlap. Cannot align traces.")

    distance_grid = np.linspace(min_common, max_common, 1000)

# ---------------------------------------------------------------
# Interpolate Throttle and Brake onto the shared distance grid.
# Telemetry CSVs from FastF1 usually have:
# - Throttle: 0..100
# - Brake: True/False or 0/1
# ---------------------------------------------------------------

throttle_aligned = {}
brake_aligned = {}

for d, df in telemetry_by_driver.items():
    # Throttle interpolation
    if 'Throttle' in df.columns:
        throttle = pd.to_numeric(df['Throttle'], errors='coerce').to_numpy(dtype=float)
        dist = df['Distance'].to_numpy(dtype=float)

        valid = ~np.isnan(throttle)
        dist_valid = dist[valid]
        thr_valid = throttle[valid]

        if len(dist_valid) >= 2:
            throttle_aligned[d] = np.interp(distance_grid, dist_valid, thr_valid)
        else:
            print(f"WARNING: Not enough throttle points for {d}.")
    else:
        print(f"WARNING: telemetry_{d}.csv missing 'Throttle'.")

    # Brake interpolation
    if 'Brake' in df.columns:
        # Brake might be boolean strings; convert to numeric 0/1
        brake_raw = df['Brake']

        # Map common boolean-like values to 0/1, then convert to float
        brake_num = brake_raw.replace({
            True: 1,
            False: 0,
            'True': 1,
            'False': 0,
            'true': 1,
            'false': 0,
        })

        brake = pd.to_numeric(brake_num, errors='coerce').to_numpy(dtype=float)
        dist = df['Distance'].to_numpy(dtype=float)

        valid = ~np.isnan(brake)
        dist_valid = dist[valid]
        br_valid = brake[valid]

        if len(dist_valid) >= 2:
            brake_aligned[d] = np.interp(distance_grid, dist_valid, br_valid)
        else:
            print(f"WARNING: Not enough brake points for {d}.")
    else:
        print(f"WARNING: telemetry_{d}.csv missing 'Brake'.")

# ---------------------------------------------------------------
# Create stacked subplots with a shared x-axis
# ---------------------------------------------------------------

fig, (ax_thr, ax_brk) = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(16, 8),
    sharex=True,
    gridspec_kw={'height_ratios': [2, 1]},
)

# Top: Throttle application
for d in drivers:
    if d not in throttle_aligned:
        continue

    ax_thr.plot(
        distance_grid,
        throttle_aligned[d],
        color=driver_colors[d],
        linewidth=1.5,
        alpha=0.9,
        label=d,
    )

add_spa_annotations(ax_thr)
ax_thr.set_title('Throttle Application')
ax_thr.set_ylabel('Throttle (%)')
ax_thr.set_ylim(-5, 105)  # Keep plot readable for 0..100 throttle
ax_thr.grid(alpha=0.2, linestyle='--')
ax_thr.legend(title='Driver', loc='upper right')

# Bottom: Brake application
for d in drivers:
    if d not in brake_aligned:
        continue

    ax_brk.plot(
        distance_grid,
        brake_aligned[d],
        color=driver_colors[d],
        linewidth=1.5,
        alpha=0.9,
        label=d,
    )

add_spa_annotations(ax_brk)
ax_brk.set_title('Brake Application')
ax_brk.set_xlabel('Distance (m)')
ax_brk.set_ylabel('Brake (0/1)')
ax_brk.set_ylim(-0.1, 1.1)
ax_brk.grid(alpha=0.2, linestyle='--')

# Overall title for the whole figure
fig.suptitle('Throttle & Brake Comparison — Fastest Lap', fontsize=16, y=0.98)

# Save chart
charts_dir = './charts'
os.makedirs(charts_dir, exist_ok=True)
thr_brk_path = os.path.join(charts_dir, 'telemetry_throttle_brake.png')
plt.savefig(thr_brk_path, dpi=150, bbox_inches='tight')
print(f"Saved telemetry throttle/brake chart to {thr_brk_path}")

plt.close(fig)

## Telemetry — Throttle & Brake (where pace is created)

Telemetry helps explain *how* lap time happens. In general, **earlier throttle application** after a corner means the car accelerates sooner, which creates more speed all the way down the next straight; even small differences add up quickly. Likewise, **later braking** usually indicates confidence and grip — the driver can carry more speed into the corner and still make the apex.

On this track, look around the **Eau Rouge / Raidillon** marker (~1000m) and into the run toward **Kemmel Straight** (~2000m): if VER’s throttle trace reaches high values earlier (or stays higher) and his brake trace drops sooner, that’s a clear sign of a stronger exit and a faster run onto the straight. That kind of repeatable “earlier throttle, stable braking” advantage is exactly what makes overtaking and sustained race pace possible over many laps.

In [ ]:
# ---------------------------------------------------------------
# SECTION 8 — Weather Context
# Short supporting section: how stable were conditions?
# ---------------------------------------------------------------

import pandas as pd               # For loading the weather CSV
import numpy as np                # For basic numerical work
import matplotlib.pyplot as plt   # For plotting
import os                         # For paths and file handling

plt.style.use('dark_background')  # Use dark style to match other charts

# ---------------------------------------------------------------
# 1) LOAD WEATHER DATA
# ---------------------------------------------------------------

weather_path = './data/weather_data.csv'          # File exported in Section 1
weather = pd.read_csv(weather_path)               # Load into a DataFrame

# Show the first few columns so we know what we have (optional for debugging)
# print(weather.head())

# ---------------------------------------------------------------
# 2) APPROXIMATE LAP NUMBERS FROM TIME SERIES
# ---------------------------------------------------------------
# The weather data is time-based (samples every few seconds). We
# want to map each weather sample to an approximate lap number for
# a 44-lap race. We do this by:
# - Creating a "sample index" from 0 .. N-1
# - Linearly mapping that index onto the range 1 .. 44
#   (this assumes weather sampling is roughly uniform in time).
# ---------------------------------------------------------------

num_samples = len(weather)                             # Total number of weather rows

affected_laps = 44                                     # Total race laps for 2022 Belgian GP

# Create a 0..N-1 index (as float) even if the DataFrame index is not 0..N-1
sample_index = np.arange(num_samples, dtype=float)

# Linearly map sample_index into the range [1, 44]
# Formula: lap ≈ 1 + (index / (N-1)) * (total_laps - 1)
if num_samples > 1:
    approx_lap = 1 + (sample_index / (num_samples - 1)) * (affected_laps - 1)
else:
    # Edge case: only one sample, treat it as mid-race
    approx_lap = np.array([affected_laps / 2.0])

weather['ApproxLap'] = approx_lap                     # Store approximate lap number in DataFrame

# ---------------------------------------------------------------
# 3) PREPARE TEMPERATURE COLUMNS
# ---------------------------------------------------------------
# FastF1 weather data usually contains columns like:
# - 'TrackTemp' for track temperature
# - 'AirTemp'   for air temperature
# We will plot track temperature vs ApproxLap, and if AirTemp
# exists, add it as a second line.
# ---------------------------------------------------------------

# Convert temperature columns to numeric (in case they were read as strings)
if 'TrackTemp' in weather.columns:
    weather['TrackTemp'] = pd.to_numeric(weather['TrackTemp'], errors='coerce')
else:
    raise RuntimeError("weather_data.csv does not contain a 'TrackTemp' column.")

has_air_temp = 'AirTemp' in weather.columns
if has_air_temp:
    weather['AirTemp'] = pd.to_numeric(weather['AirTemp'], errors='coerce')

# Drop any rows where track temperature is missing (we need it for the main line)
weather_plot = weather.dropna(subset=['TrackTemp']).copy()

# ---------------------------------------------------------------
# 4) PLOT TEMPERATURE OVER RACE DURATION
# ---------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 5))

# Track temperature line (main focus)
ax.plot(
    weather_plot['ApproxLap'],
    weather_plot['TrackTemp'],
    color='#FF6B35',
    linewidth=2.0,
    label='Track Temp',
)

# Optional: Air temperature line if available
if has_air_temp:
    # Use the same rows as track temp, but it's okay if AirTemp has a few NaNs
    air_temp_series = weather_plot['AirTemp']
    ax.plot(
        weather_plot['ApproxLap'],
        air_temp_series,
        color='#4ECDC4',
        linewidth=1.8,
        linestyle='--',
        label='Air Temp',
    )

ax.set_title('Track & Air Temperature — 2022 Belgian GP Race')
ax.set_xlabel('Approximate Lap Number')
ax.set_ylabel('Temperature (°C)')

ax.grid(alpha=0.2, linestyle='--')

# Horizontal reference line at 30°C
ax.axhline(30.0, color='white', linestyle=':', linewidth=1.2, alpha=0.8)
ax.text(
    0.01,
    30.0,
    '30°C threshold',
    color='white',
    fontsize=9,
    va='bottom',
    ha='left',
)

ax.legend(loc='best')

# Save the chart
charts_dir = './charts'
os.makedirs(charts_dir, exist_ok=True)
weather_chart_path = os.path.join(charts_dir, 'weather_temperature.png')
plt.savefig(weather_chart_path, dpi=150, bbox_inches='tight')

print(f"Saved weather temperature chart to {weather_chart_path}")

plt.close(fig)

# ---------------------------------------------------------------
# 5) AUTOMATED WEATHER SUMMARY
# ---------------------------------------------------------------
# We now:
# - Decide if the race was effectively "dry" based on Rainfall.
# - Compute max, min, and variation of track temperature.
# - Classify variation as Stable / Moderate / Significant.
# ---------------------------------------------------------------

# Check rainfall if the column exists. In FastF1 this might be
# a boolean or 0/1 column.
if 'Rainfall' in weather.columns:
    rainfall_raw = weather['Rainfall']

    # Map common representations to numeric 0/1 first
    rainfall_num = rainfall_raw.replace({
        True: 1,
        False: 0,
        'True': 1,
        'False': 0,
        'true': 1,
        'false': 0,
    })
    rainfall_num = pd.to_numeric(rainfall_num, errors='coerce').fillna(0)

    race_dry = rainfall_num.sum() == 0
else:
    # If we have no rainfall data, we assume dry for this summary
    race_dry = True

max_track_temp = float(weather_plot['TrackTemp'].max())
min_track_temp = float(weather_plot['TrackTemp'].min())
variation = max_track_temp - min_track_temp

# Decide qualitative label based on how big the variation is
if variation < 2.0:
    variation_label = 'Stable'
elif variation < 5.0:
    variation_label = 'Moderate'
else:
    variation_label = 'Significant'

print("Was the race dry?:", 'Yes' if race_dry else 'No')
print(f"Max track temp: {max_track_temp:.1f}°C")
print(f"Min track temp: {min_track_temp:.1f}°C")
print(
    f"Temperature variation of {variation:.1f}°C across the race distance. "
    f"{variation_label} conditions throughout."
)

# Keep some summary values in memory for later narrative if needed
weather_race_dry = race_dry
weather_temp_max = max_track_temp
weather_temp_min = min_track_temp
weather_temp_variation = variation
weather_temp_variation_label = variation_label

<a id="section-8"></a>
## Section 8 — Weather Context

The weather trace shows how **track and air temperature** evolve roughly lap by lap, and the summary line quantifies how big that swing really was. If the temperature variation is small and the race is effectively dry, then grip levels stay fairly constant, which means our tire degradation and correlation results from earlier sections are not being distorted by changing weather. In other words, we can trust that the regression slopes and Pearson correlations are telling a clean story about **tire behavior and driver pace**, not just reacting to a wet–dry or hot–cold track.

<a id="section-9"></a>
# Conclusion — Simply Lovely, and the Data Agrees

## What the Statistics Proved

The descriptive statistics on clean lap times showed that Verstappen combined the **lowest mean lap time** with one of the **lowest standard deviations**, a rare combination of being both the fastest and among the most consistent drivers on track. The box plots and lap-time distributions made this visible: his entire distribution of laps was shifted lower (faster) and tightly clustered compared to PER, SAI, and RUS.

The normal distribution fit and Z-score analysis treated each driver's clean laps as a probability distribution rather than just a list of times. Verstappen's mean lap time sat well below the field average in Z-score terms, meaning that laps as fast as his "typical" lap would be statistically rare if they were drawn from the other drivers' distributions. This reframes the win: it was not an outlier drive but the most likely outcome given the underlying distribution of his pace.

The tire degradation regression showed that his **slope** (seconds added per lap of tire age) was among the flattest, especially when compared to his team-mate PER. That indicates he could extend stints and push on used tires without suffering the same lap-time penalty, turning tire wear into a manageable factor rather than a limiting one.

The Pearson correlation between TyreLife and lap time confirmed how strongly each driver's pace was controlled by tire age. A lower |r| for Verstappen means his lap times were **less tightly tied** to how old the tires were, reinforcing the idea that he could keep delivering fast laps even as stints progressed, whereas others saw their pace rise more predictably with tire age.

Finally, the telemetry overlays — especially through Eau Rouge/Raidillon, onto Kemmel, and into the braking zones — showed how this advantage looked in pure driving terms: earlier throttle, confident braking, and strong speed traces translated the abstract statistics into concrete on-track behavior. The data-backed story is that Verstappen didn't just start P14 and pass people; he had the underlying pace and car control to make that progression almost inevitable.

## The Four Statistical Pillars of the Victory

| Statistical Tool                          | Finding                                                                 | Race Meaning                                                                                 |
|-------------------------------------------|-------------------------------------------------------------------------|----------------------------------------------------------------------------------------------|
| Descriptive Statistics (Section 3)        | Lowest mean lap time and low standard deviation on clean laps          | Fastest **and** most consistent driver, forming a dominant baseline pace profile            |
| Normal Distribution & Z-Score (Section 4) | Mean lap time well below field mean (strong negative Z-score)          | His "average" lap would be an unusually fast outlier for the rest of the field             |
| Tire Degradation Regression (Section 5)   | Flattest degradation slope (+s/lap) among the four analyzed drivers    | Pace faded least with tire age, enabling long, hard stints without a steep performance drop |
| Pearson Correlation (Section 6)           | Lowest correlation between TyreLife and lap time (smallest |r| value) | Tire condition constrained his pace less, allowing sustained speed regardless of tire age   |

## Final Verdict

Max Verstappen's mean lap time of XX.XXX seconds placed him
X.XX standard deviations below the field average — a statistically
dominant performance by any measure. His tire degradation rate of
+X.XXs per lap was the flattest of all four analyzed drivers, and
his Pearson correlation coefficient of X.XX confirmed that tire
condition had the least influence on his pace. The data does not
describe the victory. The data explains it.

> *"Simply Lovely."* — Max Verstappen, team radio, 2022 Belgian GP

In [ ]:
# ---------------------------------------------------------------
# FINAL SUMMARY CELL — Files and Outputs
# ---------------------------------------------------------------

import os

charts_dir = './charts'
data_dir = './data'

print("Analysis complete. All charts saved to ./charts/")
print("All data saved to ./data/\n")

print("Charts in ./charts/:")
if os.path.isdir(charts_dir):
    for fname in sorted(os.listdir(charts_dir)):
        print(f"  - {fname}")
else:
    print("  (no charts directory found)")

print("\nData files in ./data/:")
if os.path.isdir(data_dir):
    for fname in sorted(os.listdir(data_dir)):
        print(f"  - {fname}")
else:
    print("  (no data directory found)")